In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.table("medical_catalog.silver.medical_transcriptions_enriched")
print("Total rows:", df.count())
display(df.limit(5))

In [0]:
kpi_specialty = df.groupBy("medical_specialty") \
    .agg(
        count("*").alias("total_cases"),
        avg("transcription_word_count").alias("avg_word_count"),
        avg("keyword_count").alias("avg_keyword_count")
    ).orderBy(desc("total_cases"))

display(kpi_specialty)

kpi_specialty.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_specialty_summary")
print("KPI 1 saved!")

In [0]:
kpi_complexity = df.groupBy("medical_specialty", "complexity_bucket") \
    .agg(count("*").alias("case_count")) \
    .orderBy(desc("case_count"))

display(kpi_complexity)

kpi_complexity.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_complexity_distribution")
print("KPI 2 saved!")

In [0]:
kpi_keywords = df \
    .select(col("medical_specialty"), explode("keyword_array").alias("keyword")) \
    .withColumn("keyword", lower(trim(col("keyword")))) \
    .filter(col("keyword") != "") \
    .groupBy("medical_specialty", "keyword") \
    .agg(count("*").alias("frequency")) \
    .orderBy(desc("frequency"))

display(kpi_keywords.limit(20))

kpi_keywords.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_keyword_frequency")
print("KPI 3 saved!")

In [0]:
kpi_diagnosis = df \
    .filter(col("llm_diagnosis").isNotNull()) \
    .filter(~lower(col("llm_diagnosis")).isin(
        "not specified", "not mentioned", 
        "none", "unknown", "n/a"
    )) \
    .groupBy("medical_specialty", "llm_diagnosis") \
    .agg(count("*").alias("diagnosis_count")) \
    .orderBy(desc("diagnosis_count"))

display(kpi_diagnosis.limit(20))

kpi_diagnosis.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_diagnosis_frequency")
print("KPI 4 saved!")

In [0]:
kpi_medications = df \
    .select(col("medical_specialty"), explode("llm_medications").alias("medication")) \
    .withColumn("medication", lower(trim(col("medication")))) \
    .filter(col("medication") != "") \
    .groupBy("medical_specialty", "medication") \
    .agg(count("*").alias("mention_count")) \
    .orderBy(desc("mention_count"))

display(kpi_medications.limit(20))

kpi_medications.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_medication_frequency")
print("KPI 5 saved!")

In [0]:
kpi_symptoms = df \
    .select(col("medical_specialty"), explode("llm_symptoms").alias("symptom")) \
    .withColumn("symptom", lower(trim(col("symptom")))) \
    .filter(col("symptom") != "") \
    .groupBy("medical_specialty", "symptom") \
    .agg(count("*").alias("symptom_count")) \
    .orderBy(desc("symptom_count"))

display(kpi_symptoms.limit(20))

kpi_symptoms.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_symptom_frequency")
print("KPI 6 saved!")

In [0]:
kpi_length = df.groupBy("medical_specialty") \
    .agg(
        avg("transcription_word_count").alias("avg_words"),
        min("transcription_word_count").alias("min_words"),
        max("transcription_word_count").alias("max_words"),
        avg("transcription_char_length").alias("avg_chars")
    ).orderBy(desc("avg_words"))

display(kpi_length)

kpi_length.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_transcription_length")
print("KPI 7 saved!")

In [0]:
kpi_quality = df.groupBy("medical_specialty", "has_keywords") \
    .agg(count("*").alias("case_count")) \
    .withColumn("keyword_status",
        when(col("has_keywords") == True, "Has Keywords")
        .otherwise("Missing Keywords")) \
    .orderBy("medical_specialty", desc("case_count"))

display(kpi_quality)

kpi_quality.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_data_quality")
print("KPI 8 saved!")

In [0]:
kpi_med_diagnosis = df \
    .filter(col("llm_diagnosis").isNotNull()) \
    .filter(col("llm_medications").isNotNull()) \
    .filter(~lower(col("llm_diagnosis")).isin(
        "not specified", "not mentioned", "none", "unknown", "n/a"
    )) \
    .select(
        col("medical_specialty"),
        col("llm_diagnosis"),
        explode("llm_medications").alias("medication")
    ) \
    .withColumn("medication", lower(trim(col("medication")))) \
    .filter(col("medication") != "") \
    .groupBy("medical_specialty", "llm_diagnosis", "medication") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count"))

display(kpi_med_diagnosis.limit(20))

kpi_med_diagnosis.write.format("delta").mode("overwrite") \
    .saveAsTable("medical_catalog.gold.kpi_medication_per_diagnosis")
print("KPI 9 saved!")

In [0]:
kpi_case_types = df.groupBy("medical_specialty") \
    .agg(
        count("*").alias("total_cases"),
        sum("keyword_count").alias("total_keywords"),
        avg("keyword_count").alias("avg_keywords")
    ).orderBy(desc("total_cases"))

display(kpi_case_types)

kpi_case_types.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.gold.kpi_case_types")
print("KPI 10 saved!")

In [0]:
display(spark.sql("SHOW TABLES IN medical_catalog.gold"))